# Machine Learning Experiment

This notebook demonstrates a complete ML workflow:
1. Data preparation
2. Model training
3. Evaluation
4. Model comparison

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

print("✅ Libraries loaded successfully!")

## 1. Generate Dataset

In [ ]:
# Generate synthetic dataset
X, y = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    n_classes=2,
    random_state=42,
    flip_y=0.1  # Add 10% noise
)

# Create DataFrame
feature_names = [f'feature_{i+1}' for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{df['target'].value_counts()}")
print(f"Class balance: {df['target'].value_counts(normalize=True).round(3)}")

## 2. Data Preparation

In [ ]:
# Split features and target
X = df.drop('target', axis=1)
y = df['target']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

## 3. Model Training & Comparison

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=5)
}

# Train and evaluate models
results = []

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred_train = model.predict(X_train_scaled)
    y_pred_test = model.predict(X_test_scaled)
    
    # Probabilities
    y_proba_train = model.predict_proba(X_train_scaled)[:, 1]
    y_proba_test = model.predict_proba(X_test_scaled)[:, 1]
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
    
    # Calculate metrics
    train_acc = (y_pred_train == y_train).mean()
    test_acc = (y_pred_test == y_test).mean()
    train_auc = roc_auc_score(y_train, y_proba_train)
    test_auc = roc_auc_score(y_test, y_proba_test)
    
    results.append({
        'Model': name,
        'Train Accuracy': train_acc,
        'Test Accuracy': test_acc,
        'Train AUC': train_auc,
        'Test AUC': test_auc,
        'CV Score (mean)': cv_scores.mean(),
        'CV Score (std)': cv_scores.std()
    })
    
    print(f"  Test Accuracy: {test_acc:.4f}")
    print(f"  Test AUC: {test_auc:.4f}")
    print(f"  CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Results DataFrame
results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("Model Comparison")
print("="*80)
results_df

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
results_df.plot(x='Model', y=['Train Accuracy', 'Test Accuracy'], 
                kind='bar', ax=axes[0], rot=45)
axes[0].set_title('Model Accuracy Comparison')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim([0.7, 1.0])
axes[0].legend(['Train', 'Test'])
axes[0].grid(axis='y')

# AUC comparison
results_df.plot(x='Model', y=['Train AUC', 'Test AUC'], 
                kind='bar', ax=axes[1], rot=45)
axes[1].set_title('Model AUC Comparison')
axes[1].set_ylabel('AUC Score')
axes[1].set_ylim([0.7, 1.0])
axes[1].legend(['Train', 'Test'])
axes[1].grid(axis='y')

plt.tight_layout()
plt.show()

## 4. Detailed Evaluation of Best Model

In [ ]:
# Select best model (highest test AUC)
best_model_name = results_df.loc[results_df['Test AUC'].idxmax(), 'Model']
best_model = models[best_model_name]

print(f"Best Model: {best_model_name}")
print("\nClassification Report:")
print(classification_report(y_test, best_model.predict(X_test_scaled)))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, best_model.predict(X_test_scaled))

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', square=True)
plt.title(f'Confusion Matrix - {best_model_name}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# ROC Curve
plt.figure(figsize=(8, 6))

for name, model in models.items():
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Model Comparison')
plt.legend()
plt.grid(True)
plt.show()

## 5. Feature Importance (if available)

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    # Get feature importances
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    # Plot
    plt.figure(figsize=(12, 6))
    plt.title(f'Feature Importance - {best_model_name}')
    plt.bar(range(min(15, len(importances))), importances[indices[:15]])
    plt.xticks(range(min(15, len(importances))), 
               [feature_names[i] for i in indices[:15]], rotation=45, ha='right')
    plt.xlabel('Feature')
    plt.ylabel('Importance')
    plt.tight_layout()
    plt.show()
else:
    print(f"Feature importance not available for {best_model_name}")

## 6. Save Model (Optional)

In [ ]:
# Uncomment to save the model
# import joblib
# joblib.dump(best_model, '../models/best_model.pkl')
# joblib.dump(scaler, '../models/scaler.pkl')
# print("Model saved successfully!")

print("Experiment complete!")